In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

data = pd.read_csv("heart (1).csv")

target_col = data.columns[-1]

X = data.drop(target_col, axis=1)
y = data[target_col]

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42)
}

print("=== Without PCA ===")
results = {}

for name, model in models.items():
    pipe = Pipeline([
        ("pre", preprocessor),
        ("clf", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    results[name] = acc
    print(f"{name} Accuracy: {acc:.4f}")

best_model_name = max(results, key=results.get)
print("\nBest Model:", best_model_name)

print("\n=== With PCA ===")
results_pca = {}

for name, model in models.items():
    pipe = Pipeline([
        ("pre", preprocessor),
        ("pca", PCA(n_components=0.95)),
        ("clf", model)
    ])

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    results_pca[name] = acc
    print(f"{name} Accuracy (PCA): {acc:.4f}")

best_model_pca = max(results_pca, key=results_pca.get)
print("\nBest Model after PCA:", best_model_pca)

=== Without PCA ===
SVM Accuracy: 0.8967
Logistic Regression Accuracy: 0.8859
Random Forest Accuracy: 0.8913

Best Model: SVM

=== With PCA ===
SVM Accuracy (PCA): 0.8967
Logistic Regression Accuracy (PCA): 0.8967
Random Forest Accuracy (PCA): 0.8750

Best Model after PCA: SVM
